In [13]:
! pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 79.9 MB/s eta 0:00:00:00:0100:01


In [31]:
import torch
from sklearn.model_selection import train_test_split


import re
import pandas as pd
import numpy as np


from gensim.models import Word2Vec
from collections import Counter


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [32]:
path = '/content/drive/My Drive/Sentence_pairs_in_English_Hindi.tsv'

df = pd.read_csv(path, sep='\t')
print(df.head())

   1282                       Muiriel is 20 now.   485968  \
0  1282                       Muiriel is 20 now.  2060319   
1  1294  Education in this world disappoints me.   485564   
2  1302                       That won't happen.  2060320   
3  1308                              I miss you.  2060321   
4  1308                              I miss you.  2060322   

              म्यूरियल अब बीस साल की हो गई है।  
0                   म्यूरियल अब बीस साल की है।  
1  मैं इस दुनिया में शिक्षा पर बहुत निराश हूँ।  
2                              वैसा नहीं होगा।  
3                 मुझें तुम्हारी याद आ रही है।  
4                     मुझें आपकी याद आ रही है।  


In [16]:
English=df.iloc[:,1]
Hindi=df.iloc[:,3]
print(English)
print(Hindi)

0                             Muiriel is 20 now.
1        Education in this world disappoints me.
2                             That won't happen.
3                                    I miss you.
4                                    I miss you.
                          ...                   
13759                      I don't want to walk.
13760                         Everyone sit down.
13761                       Even Tom was crying.
13762       Tom doesn't need to work. He's rich.
13763                      I have three cameras.
Name: Muiriel is 20 now., Length: 13764, dtype: object
0                            म्यूरियल अब बीस साल की है।
1           मैं इस दुनिया में शिक्षा पर बहुत निराश हूँ।
2                                       वैसा नहीं होगा।
3                          मुझें तुम्हारी याद आ रही है।
4                              मुझें आपकी याद आ रही है।
                              ...                      
13759                              मैं चलना नहीं चाहती।
13760         

In [33]:
English_temp, English_test, Hindi_temp, Hindi_test = train_test_split(English, Hindi, test_size=0.2, random_state=42)

English_train, English_val, Hindi_train, Hindi_val = train_test_split(English_temp, Hindi_temp, test_size=0.25, random_state=42)

In [34]:
def tokenize(text):
    return re.findall(r'\b\w+\b', text.lower())

In [35]:
special_tokens = ['<PAD>', '<UNK>']
MIN_FREQ = 2
embedding_dim=300

Creating Embedding for English

In [40]:
English_train_sentences = [tokenize(x) for x in English_train]
print(len(English_train_sentences))

English_counter = Counter()
for s in English_train_sentences:
    English_counter.update(s)

English_valid_words = [w for w, count in English_counter.most_common() if count >= MIN_FREQ]


English_word2idx={tok: idx for idx, tok in enumerate(special_tokens)}
for word in English_valid_words:
    English_word2idx[word] = len(English_word2idx)

English_idx2word = {idx: word for word, idx in English_word2idx.items()}

English_word2vec = Word2Vec(
    sentences=English_train_sentences,
    vector_size=embedding_dim,
    window=5,
    min_count=2,
    workers=4,
    sg=1
)

English_embedding_matrix = [np.zeros(embedding_dim), np.random.normal(scale=0.6, size=(embedding_dim,))]
for word in English_valid_words:
    if word in English_word2vec.wv:
        English_embedding_matrix.append(English_word2vec.wv[word])
    else:
        English_embedding_matrix.append(np.random.normal(scale=0.6, size=(embedding_dim,)))

English_embedding_matrix = torch.from_numpy(np.array(English_embedding_matrix)).float()

if len(English_embedding_matrix)==len(English_word2idx)==len(English_idx2word):
    print(len(English_word2idx))
    print(English_embedding_matrix.shape)

8258
2230
torch.Size([2230, 300])


Creating Embedding for Hindi

In [41]:
Hindi_train_sentences = [tokenize(x) for x in Hindi_train]
print(len(Hindi_train_sentences))

Hindi_counter = Counter()
for s in Hindi_train_sentences:
    Hindi_counter.update(s)

Hindi_valid_words = [w for w, count in Hindi_counter.most_common() if count >= MIN_FREQ]

Hindi_word2idx={tok: idx for idx, tok in enumerate(special_tokens)}
for word in Hindi_valid_words:
    Hindi_word2idx[word] = len(Hindi_word2idx)

Hindi_idx2word = {idx: word for word, idx in Hindi_word2idx.items()}


Hindi_word2vec = Word2Vec(
    sentences=Hindi_train_sentences,
    vector_size=embedding_dim,
    window=5,
    min_count=2,
    workers=4,
    sg=1
)

Hindi_embedding_matrix = [np.zeros(embedding_dim), np.random.normal(scale=0.6, size=(embedding_dim,))]
for word in Hindi_valid_words:
    if word in Hindi_word2vec.wv:
        Hindi_embedding_matrix.append(Hindi_word2vec.wv[word])
    else:
        Hindi_embedding_matrix.append(np.random.normal(scale=0.6, size=(embedding_dim,)))

Hindi_embedding_matrix = torch.from_numpy(np.array(Hindi_embedding_matrix)).float()

if len(Hindi_embedding_matrix)==len(Hindi_word2idx)==len(Hindi_idx2word):
    print(len(Hindi_word2idx))
    print(Hindi_embedding_matrix.shape)

8258
745
torch.Size([745, 300])


Creating Custum DataSet and DataLoaders

In [42]:
print(2+5)

7
